In [ ]:
import pandas as pd

# 1. EXTRACT
df = pd.read_csv('data/raw/combined_stock_data.csv')
df_copy = df.copy()

print("Initial missing values:")
print(df_copy.isnull().sum())

# 2. TRANSFORM - Cleaning
df_copy.drop(columns=['Price'], inplace=True) # Fixed this line
df_copy.dropna(subset=['Date'], inplace=True)
df_copy['Date'] = pd.to_datetime(df_copy['Date'])
df_copy = df_copy.sort_values(['Company Name', 'Date'])

# 2. TRANSFORM - Imputation
df_copy['Open'] = df_copy['Open'].fillna(df_copy.groupby('Company Name')['Close'].shift(1))
df_copy['Close'] = df_copy['Close'].fillna(df_copy.groupby('Company Name')['Open'].shift(-1))
df_copy['High'] = df_copy.groupby('Company Name')['High'].ffill()
df_copy['Low'] = df_copy.groupby('Company Name')['Low'].ffill()

df_copy['Year'] = df_copy['Date'].dt.year
df_copy['Volume'] = df_copy['Volume'].fillna(df_copy.groupby(['Company Name', 'Year'])['Volume'].transform('median'))
df_copy['Volume'] = df_copy['Volume'].fillna(0)

df_copy['Month'] = df_copy['Date'].dt.month
df_copy['Adj Close'] = df_copy['Adj Close'].fillna(df_copy.groupby(['Company Name', 'Month', 'Year'])['Adj Close'].transform('median'))
df_copy['Adj Close'] = df_copy['Adj Close'].fillna(0)

print("\nMissing values after imputation:")
print(df_copy.isnull().sum())

# 2. TRANSFORM - Feature Engineering
df_copy['Day_Range'] = df_copy['High'] - df_copy['Low']
df_copy['Daily_Return'] = df_copy.groupby('Company Name')['Close'].pct_change() * 100

# 3. LOAD
output_path = 'data/processed/cleaned_stock_data.csv'
df_copy.to_csv(output_path, index=False)
print(f"\nETL pipeline complete! Clean data saved to {output_path}")